# Kernel Bandwidth on a T4

Tests the second CUDA kernel and measures every version of the index on
the random benchmark input and on a real ERA5 field
([docs/gpu-performance.md](../docs/gpu-performance.md#the-gap-to-peak-bandwidth)).
The changes and their success criteria were written before this
notebook was run.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU and takes about 20 minutes.

The notebook gets the project from GitHub. To use a local copy instead,
upload the repository as a Kaggle dataset named `earth-intelligence`
and add it to the notebook.

Output: `/kaggle/working/kernel_bandwidth.zip`. Unzip it into
`ml/artifacts/gpu_perf/`.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import shutil
import subprocess
from pathlib import Path

import torch
import triton

print("torch", torch.__version__, "cuda", torch.version.cuda)
print("triton", triton.__version__)
print("compute capability", torch.cuda.get_device_capability(0))

## Project

In [ ]:
REPOSITORY = "https://github.com/khoahonguyen0210/Earth_intelligence.git"
INPUT = Path("/kaggle/input")
PROJECT = Path("/kaggle/working/earth-intelligence")
OUTPUT = Path("/kaggle/working/kernel_bandwidth")
ERA5 = Path("/kaggle/working/era5_columns.pt")


def uploaded_project():
    for marker in INPUT.rglob("ml/pyproject.toml"):
        return marker.parent.parent
    for archive in INPUT.rglob("earth-intelligence.tar.gz"):
        shutil.unpack_archive(archive, PROJECT)
        return PROJECT
    return None


if not PROJECT.exists():
    source = uploaded_project()
    if source is None:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True
        )
    elif source != PROJECT:
        shutil.copytree(source, PROJECT)
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q ninja pytest python-dotenv "earth2studio[data]==0.20.0" 2>&1 | tail -3

## Tests

All five versions against the float64 reference, and the second CUDA
kernel against the first, bit for bit.

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
!cd {PROJECT}/ml && python -m pytest -q tests/kernels 2>&1 | tee {OUTPUT}/tests.txt | tail -15

## ERA5 field

Downloads ERA5 at 2025-08-03 06:00 UTC on the 13 FCN3 levels from ARCO.

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/fetch_era5_columns.py --output {ERA5}

## Benchmark

Measures the T4's peak bandwidth with a device-to-device copy, then
times every version at three sizes, first on the random input and then
on the ERA5 field.

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/run_gpu_benchmark.py --output {OUTPUT}

In [ ]:
!cd {PROJECT} && PYTHONPATH=ml/src python scripts/research/run_gpu_benchmark.py --output {OUTPUT} --era5 {ERA5}

In [ ]:
ERA5.unlink(missing_ok=True)
shutil.make_archive("/kaggle/working/kernel_bandwidth", "zip", OUTPUT)
sorted(path.name for path in OUTPUT.rglob("*"))